## Model 1: Random Forest Classifier

### 1. Model Selection Rationale
* **Dataset Characteristics**: The dataset contains a mix of numerical and categorical features related to student demographics, academic background, and social-economic factors. Random Forest handles heterogeneous data exceptionally well without requiring extensive feature scaling.
* **Problem Type**: This is a multi-class/binary classification problem aimed at predicting student outcomes (Dropout, Enrolled, Graduate). Random Forest natively supports multi-class classification and handles complex non-linear relationships.
* **Strengths**:
  * High accuracy and robustness against overfitting through ensemble learning (bagging).
  * Built-in feature importance evaluation, which helps leadership identify key factors influencing student success.
  * Less sensitive to outliers and missing values compared to other algorithms.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Load the preprocessed dataset
df = pd.read_csv('../Dataset/preprocessed_data.csv')

# 2. Separate features (X) and target (y)
X = df.drop(columns=['Target'])
y = df['Target']

# 3. Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Train the baseline Random Forest model
rf_base = RandomForestClassifier(random_state=42)
rf_base.fit(X_train, y_train)

# Check baseline accuracy
y_pred_base = rf_base.predict(X_test)
print("Baseline Accuracy:", accuracy_score(y_test, y_pred_base))

# 5. Hyperparameter tuning using GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# Print best parameters and score
print("Best Parameters:", grid_search.best_params_)
print("Best CV Accuracy:", grid_search.best_score_)

# 6. Evaluate the tuned model on the test set
best_rf = grid_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test)
print("Tuned Test Accuracy:", accuracy_score(y_test, y_pred_tuned))

# 7. Plot and save feature importance
importances = best_rf.feature_importances_
feature_imp_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_imp_df, palette='viridis')
plt.title('Top 10 Most Important Features - Random Forest')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.tight_layout()
plt.savefig('../Supervised_Learning/plots/rf_feature_importance.png')
plt.show()